# Receipt field extraction: LoRA fine-tuning of a small LM (Avathon Track B / S2)

Runtime: **GPU (T4)**. Run top to bottom. Outputs land in `preds/`, `results/` and `runs/` - commit the small files (`preds/`, `results/`, `runs/*/run_info.json`), not the weights.

Order: setup, data, baselines (no training), train, evaluate, ablations, forgetting check, error analysis, latency, export.

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-3e25adc2-645a-b0b0-9f51-1994b76d9630)


In [ ]:
REPO_URL = "https://github.com/ai-pp-2025/avathon-slm-doc-extraction.git"
!git clone -q {REPO_URL} repo
%cd repo
!pip -q install -r requirements.txt
!pip -q install -e . --no-deps

/content/repo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for docextract (pyproject.toml) ... done


In [3]:
!pwd

/content/repo


In [4]:
!ls

data	   preds	   README.md	     src    WRITEUP.md
notebooks  pyproject.toml  requirements.txt  tests


In [6]:
# Downloads SROIE (~500 MB, includes images we drop), builds merchant-grouped train/val and the held-out test split.
!python -m docextract.data --out data/processed
!mkdir -p results && cp data/processed/data_card.json results/

README.md: 100% 14.3k/14.3k [00:00<00:00, 25.5MB/s]

data/train-00000-of-00001.parquet: downloading bytes:  39% 124M/319M [00:01<00:01, 194MB/s, 7.89MB/s  ] 
data/train-00000-of-00001.parquet: downloading bytes:  90% 287M/319M [00:02<00:00, 172MB/s, 22.9MB/s  ]
data/train-00000-of-00001.parquet: reconstructing file:  55% 176M/319M [00:02<00:01, 72.8MB/s, 6.48MB/s  ]
data/train-00000-of-00001.parquet: downloading bytes: 100% 308M/308M [00:06<00:00, 46.8MB/s, 27.4MB/s  ]
data/train-00000-of-00001.parquet: reconstructing file: 100% 319M/319M [00:06<00:00, 48.4MB/s, 23.4MB/s  ]

data/test-00000-of-00001.parquet: downloading bytes:  35% 67.5M/191M [00:01<00:01, 95.4MB/s, 4.70MB/s  ]
data/test-00000-of-00001.parquet: downloading bytes:  73% 140M/191M [00:01<00:00, 163MB/s, 10.4MB/s  ]
data/test-00000-of-00001.parquet: reconstructing file:  66% 126M/191M [00:01<00:00, 74.9MB/s, 6.59MB/s  ]
data/test-00000-of-00001.parquet: downloading bytes:  91% 174M/191M [00:02<00:00, 135MB/s, 14.8MB/s  ]
d

## 1. Baselines (no training)
What the fine-tuned model has to beat: a rule-based extractor, the same base model prompted zero/few-shot, and (optional) a 7B model prompted in 4-bit.

In [7]:
pwd

'/content/repo'

In [8]:
!ls ../../../content

repo  sample_data


In [9]:
!ls

data	   preds	   README.md	     results  tests
notebooks  pyproject.toml  requirements.txt  src      WRITEUP.md


In [10]:
!python -m docextract.infer --method regex --out preds/regex.jsonl
!python -m docextract.infer --model Qwen/Qwen3-0.6B --shots 0 --out preds/qwen3-0.6b_zeroshot.jsonl
!python -m docextract.infer --model Qwen/Qwen3-0.6B --shots 3 --batch-size 4 --out preds/qwen3-0.6b_3shot.jsonl

wrote 361 predictions to preds/regex.jsonl; fields=('company', 'date', 'total')
config.json: 100% 726/726 [00:00<00:00, 2.75MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 12.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 56.2MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 126MB/s]

tokenizer.json: downloading bytes:   0% 0.00/11.4M [00:00<?, ?B/s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:00<00:00, 6.93MB/s,  336kB/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:00<00:00, 23.3MB/s, 1.13MB/s  ]

model.safetensors: downloading bytes:   4% 59.3M/1.50G [00:00<00:12, 114MB/s, 2.10MB/s  ]
model.safetensors: downloading bytes:  52% 782M/1.50G [00:03<00:02, 247MB/s, 63.8MB/s  ]
model.safetensors: downloading bytes:  76% 1.15G/1.50G [00:05<00:02, 129MB/s, 81.9MB/s  ]
model.safetensors: reconstructing file:  45% 679M/1.50G [00:05<00:06, 133MB/s, 23.2MB/s  ] 
model.safetensors: downloading bytes:  82% 1.24G/1.50G [00:06<00:01, 172MB/s, 85.5MB/s  ]
mod

## 2. Fine-tune (main run)
LoRA r=16, alpha=32, all linear layers, lr 2e-4, 3 epochs. Hyper-parameters are logged to `runs/<name>/train_args.json`, cost to `run_info.json`, loss curves to `log_history.json`.

Run the 16-sample smoke test first if you changed anything.

In [12]:
!pip install -U torchao>=0.16.0

In [13]:
import torchao
print(torchao.__version__)

0.18.0


In [19]:
!python -m docextract.train --model Qwen/Qwen3-0.6B --rank 16 --out runs/smoke --batch-size 4 --max-train-samples 16 --epochs 1 --grad-accum 4 --grad-ckpt

train=16 val=75 max_len=599
Loading weights: 100% 311/311 [00:00<00:00, 5401.54it/s]
trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
100% 1/1 [00:18<00:00, 18.25s/it]
  0% 0/19 [00:00<?, ?it/s]
 11% 2/19 [00:00<00:08,  2.11it/s]
 16% 3/19 [00:01<00:10,  1.47it/s]
 21% 4/19 [00:03<00:13,  1.12it/s]
 26% 5/19 [00:04<00:15,  1.10s/it]
 32% 6/19 [00:05<00:14,  1.12s/it]
 37% 7/19 [00:07<00:14,  1.19s/it]
 42% 8/19 [00:08<00:13,  1.20s/it]
 47% 9/19 [00:09<00:11,  1.17s/it]
 53% 10/19 [00:11<00:11,  1.32s/it]
 58% 11/19 [00:12<00:11,  1.39s/it]
 63% 12/19 [00:14<00:09,  1.37s/it]
 68% 13/19 [00:15<00:08,  1.35s/it]
 74% 14/19 [00:16<00:06,  1.34s/it]
 79% 15/19 [00:17<00:05,  1.34s/it]
 84% 16/19 [00:19<00:04,  1.38s/it]
 89% 17/19 [00:20<00:02,  1.38s/it]
 95% 18/19 [00:22<00:01,  1.35s/it]
                                 
{'eval_loss': '0.7565', 'eval_runtime': '24.32', 'eval_samples_per_second': '3.084', 'eval_steps_per_second': '0.781', 'epoch': '1'}
100% 

In [20]:
!python -m docextract.train --model Qwen/Qwen3-0.6B --rank 16 --out runs/qwen3-0.6b_lora_r16 --grad-accum 4 --grad-ckpt

train=551 val=75 max_len=875
Loading weights: 100% 311/311 [00:00<00:00, 6507.18it/s]
trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
{'loss': '0.3384', 'grad_norm': '0.7365', 'learning_rate': '0.0001992', 'epoch': '0.2899'}
{'loss': '0.04168', 'grad_norm': '0.446', 'learning_rate': '0.0001905', 'epoch': '0.5797'}
{'loss': '0.02561', 'grad_norm': '0.2688', 'learning_rate': '0.0001729', 'epoch': '0.8696'}
 33% 35/105 [10:51<18:52, 16.18s/it]
  0% 0/19 [00:00<?, ?it/s]
 11% 2/19 [00:01<00:08,  1.99it/s]
 16% 3/19 [00:02<00:11,  1.39it/s]
 21% 4/19 [00:03<00:14,  1.06it/s]
 26% 5/19 [00:04<00:16,  1.15s/it]
 32% 6/19 [00:06<00:15,  1.18s/it]
 37% 7/19 [00:07<00:15,  1.25s/it]
 42% 8/19 [00:08<00:13,  1.26s/it]
 47% 9/19 [00:09<00:12,  1.23s/it]
 53% 10/19 [00:11<00:12,  1.38s/it]
 58% 11/19 [00:13<00:11,  1.46s/it]
 63% 12/19 [00:14<00:10,  1.44s/it]
 68% 13/19 [00:16<00:08,  1.43s/it]
 74% 14/19 [00:17<00:07,  1.42s/it]
 79% 15/19 [00:18<00:05,  1.42s/it]
 8

In [21]:
!python -m docextract.infer --model Qwen/Qwen3-0.6B --adapter runs/qwen3-0.6b_lora_r16/adapter --out preds/qwen3-0.6b_lora_r16.jsonl

Loading weights: 100% 311/311 [00:00<00:00, 4711.68it/s]
8/361
16/361
24/361
32/361
40/361
48/361
56/361
64/361
72/361
80/361
88/361
96/361
104/361
112/361
120/361
128/361
136/361
144/361
152/361
160/361
168/361
176/361
184/361
192/361
200/361
208/361
216/361
224/361
232/361
240/361
248/361
256/361
264/361
272/361
280/361
288/361
296/361
304/361
312/361
320/361
328/361
336/361
344/361
352/361
360/361
361/361
wrote 361 predictions to preds/qwen3-0.6b_lora_r16.jsonl; fields=('company', 'date', 'total')


In [22]:
!python -m docextract.evaluate preds/*.jsonl --out results/summary.md

| run | n | JSON ok | company EM | company F1 | date EM | total EM | all3 EM | all3 EM (unseen merchants) | ungrounded | gold ungrounded | ms/receipt |
|---|---|---|---|---|---|---|---|---|---|---|---|
| qwen3-0.6b_3shot | 361 | 100.0% | 69.3% | 87.9% | 91.1% | 73.7% | 54.0% | 40.0% | 2.8% | 2.8% | 2350 |
| qwen3-0.6b_lora_r16 | 361 | 100.0% | 93.4% | 96.4% | 98.3% | 95.8% | 88.1% | 73.6% | 0.0% | 2.8% | 789 |
| qwen3-0.6b_zeroshot | 361 | 100.0% | 66.8% | 87.2% | 63.7% | 80.9% | 37.1% | 32.0% | 8.0% | 2.8% | 902 |
| regex | 361 | 100.0% | 64.0% | 90.7% | 95.0% | 71.7% | 46.5% | 35.2% | 0.0% | 2.8% | 0 |


## 3. Ablations (the evidence behind each design choice)
Each row answers one question in the write-up: LoRA rank, target modules, LoRA vs QLoRA, learning curve (learned vs memorised), base-model choice. Comment out rows you do not have time for.

In [24]:
# (run name, base model, extra train flags)
ABLATIONS = [
    ("qwen3-0.6b_lora_r4_n100",        "Qwen/Qwen3-0.6B", "--rank 4 --max-train-samples 100 --grad-accum 8 --grad-ckpt"),
    ("qwen3-0.6b_lora_r64_n100",       "Qwen/Qwen3-0.6B", "--rank 64 --max-train-samples 100 --grad-accum 8 --grad-ckpt"),
    ("qwen3-0.6b_lora_attn_r16_n100",  "Qwen/Qwen3-0.6B", "--rank 16 --targets attn --max-train-samples 100 --grad-accum 8 --grad-ckpt"),
    ("qwen3-0.6b_qlora_r16_n100",      "Qwen/Qwen3-0.6B", "--rank 16 --qlora --max-train-samples 100 --grad-accum 8 --grad-ckpt"),
    ("qwen3-0.6b_lora_r16_n100",  "Qwen/Qwen3-0.6B", "--max-train-samples 100 --grad-accum 8 --grad-ckpt"),
    ("qwen3-0.6b_lora_r16_n300",  "Qwen/Qwen3-0.6B", "--max-train-samples 300 --grad-accum 8 --grad-ckpt"),
    # ("qwen3-1.7b_lora_r16",       "Qwen/Qwen3-1.7B", "--batch-size 2 --grad-accum 8 --grad-ckpt"),
    # ("qwen2.5-0.5b_lora_r16",     "Qwen/Qwen2.5-0.5B-Instruct", ""),
    # ("smollm2-1.7b_lora_r16",     "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--batch-size 2 --grad-accum 8 --grad-ckpt"),
]
for name, model, flags in ABLATIONS:
    !python -m docextract.train --model {model} --out runs/{name} {flags}
    !python -m docextract.infer --model {model} --adapter runs/{name}/adapter --out preds/{name}.jsonl

train=100 val=75 max_len=724
Loading weights: 100% 311/311 [00:00<00:00, 5188.62it/s]
trainable params: 2,523,136 || all params: 598,573,056 || trainable%: 0.4215
 33% 4/12 [01:42<02:53, 21.71s/it]
  0% 0/19 [00:00<?, ?it/s]
 11% 2/19 [00:00<00:07,  2.14it/s]
 16% 3/19 [00:01<00:10,  1.49it/s]
 21% 4/19 [00:03<00:13,  1.13it/s]
 26% 5/19 [00:04<00:15,  1.08s/it]
 32% 6/19 [00:05<00:14,  1.10s/it]
 37% 7/19 [00:07<00:14,  1.17s/it]
 42% 8/19 [00:08<00:13,  1.18s/it]
 47% 9/19 [00:09<00:11,  1.15s/it]
 53% 10/19 [00:10<00:11,  1.30s/it]
 58% 11/19 [00:12<00:10,  1.37s/it]
 63% 12/19 [00:13<00:09,  1.35s/it]
 68% 13/19 [00:15<00:08,  1.34s/it]
 74% 14/19 [00:16<00:06,  1.33s/it]
 79% 15/19 [00:17<00:05,  1.33s/it]
 84% 16/19 [00:19<00:04,  1.36s/it]
 89% 17/19 [00:20<00:02,  1.36s/it]
 95% 18/19 [00:21<00:01,  1.34s/it]
                                  
{'eval_loss': '0.488', 'eval_runtime': '23.99', 'eval_samples_per_second': '3.127', 'eval_steps_per_second': '0.792', 'epoch': '1'}
 33%

In [25]:
!python -m docextract.evaluate preds/*.jsonl --out results/summary.md

| run | n | JSON ok | company EM | company F1 | date EM | total EM | all3 EM | all3 EM (unseen merchants) | ungrounded | gold ungrounded | ms/receipt |
|---|---|---|---|---|---|---|---|---|---|---|---|
| qwen3-0.6b_3shot | 361 | 100.0% | 69.3% | 87.9% | 91.1% | 73.7% | 54.0% | 40.0% | 2.8% | 2.8% | 2350 |
| qwen3-0.6b_lora_attn_r16_n100 | 361 | 100.0% | 81.7% | 91.5% | 94.7% | 90.9% | 71.5% | 50.4% | 3.9% | 2.8% | 794 |
| qwen3-0.6b_lora_r16 | 361 | 100.0% | 93.4% | 96.4% | 98.3% | 95.8% | 88.1% | 73.6% | 0.0% | 2.8% | 789 |
| qwen3-0.6b_lora_r16_n100 | 361 | 100.0% | 83.9% | 91.0% | 96.4% | 91.4% | 74.5% | 52.0% | 1.1% | 2.8% | 808 |
| qwen3-0.6b_lora_r16_n300 | 361 | 100.0% | 92.2% | 96.1% | 97.0% | 94.5% | 85.0% | 68.8% | 0.3% | 2.8% | 789 |
| qwen3-0.6b_lora_r4_n100 | 361 | 100.0% | 81.4% | 90.8% | 93.9% | 87.5% | 70.1% | 46.4% | 3.0% | 2.8% | 792 |
| qwen3-0.6b_lora_r64_n100 | 361 | 100.0% | 84.2% | 91.0% | 95.6% | 91.4% | 74.2% | 58.4% | 0.8% | 2.8% | 795 |
| qwen3-0.6b_qlora_r16

## 4. Catastrophic forgetting + latency
Perplexity on WikiText-2 with the adapter off vs on; then single-receipt latency (batch size 1) on 50 receipts.

In [26]:
!python -m docextract.forgetting --model Qwen/Qwen3-0.6B --adapter runs/qwen3-0.6b_lora_r16/adapter --out results/forgetting_qwen3-0.6b_lora_r16.json
!python -m docextract.infer --model Qwen/Qwen3-0.6B --adapter runs/qwen3-0.6b_lora_r16/adapter --batch-size 1 --limit 50 --out results/latency/qwen3-0.6b_lora_r16_bs1.jsonl
!python -m docextract.evaluate results/latency/qwen3-0.6b_lora_r16_bs1.jsonl

README.md: 100% 10.5k/10.5k [00:00<00:00, 12.7MB/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/repo/src/docextract/forgetting.py", line 52, in <module>
    main()
    ~~~~^^
  File "/content/repo/src/docextract/forgetting.py", line 24, in main
    texts = [t for t in load_dataset("wikitext", "wikitext-2-raw-v1", split="test")["text"] if len(t.split()) > 50]
                        ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/datasets/load.py", line 1688, in load_dataset
    builder_instance = load_dataset_builder(
        path=path,
    ...<10 lines>...
        **config_kwargs,
    )
  File "/usr/local/lib/python3.13/dist-packages/datasets/load.py", line 1315, in load_dataset_builder
    dataset_module = dataset_module_factory(
        path,
    ...<5 lines>...
        cache_dir=cache_dir,
    )
  File "/usr/loc

## 5. Qualitative comparison and error analysis

In [30]:
!pwd

/content/repo


In [31]:
import json
from src.docextract.data import load_jsonl
from src.docextract.metrics import FIELDS, score_example

test = {r["key"]: r for r in load_jsonl("data/processed/test.jsonl")}
base = {r["key"]: r for r in load_jsonl("preds/qwen3-0.6b_3shot.jsonl")}
ft = {r["key"]: r for r in load_jsonl("preds/qwen3-0.6b_lora_r16.jsonl")}

# Same receipts, base (3-shot) vs fine-tuned: pick cases where they differ.
shown = 0
for k, f in ft.items():
    b = base[k]
    if f["pred"] != b["pred"]:
        print(k, "\n  gold :", f["gold"], "\n  base :", b["raw"].strip()[:200], "\n  tuned:", f["raw"].strip()[:200], "\n")
        shown += 1
    if shown == 5:
        break

X00016469670 
  gold : {'company': 'OJC MARKETING SDN BHD', 'date': '15/01/2019', 'total': '193.00'} 
  base : {"company": "TAN CHAY YEE", "date": "15/01/2019", "total": "193.00"} 
  tuned: {"company": "OJC MARKETING SDN BHD", "date": "15/01/2019", "total": "193.00"} 

X00016469671 
  gold : {'company': 'OJC MARKETING SDN BHD', 'date': '02/01/2019', 'total': '170.00'} 
  base : {"company": "TAN CHAY YEE", "date": "02/01/2019 2:47:14 PM", "total": "170.00"} 
  tuned: {"company": "OJC MARKETING SDN BHD", "date": "02/01/2019", "total": "170.00"} 

X51005230605 
  gold : {'company': 'PETRON BKT LANJAN SB', 'date': '01/02/2018', 'total': '4.90'} 
  base : {"company": "PETRON BKT LANJAN SB", "date": "01/02/2018", "total": "4.90"} 
  tuned: {"company": "ALSERKAM ENTERPRISE", "date": "01/02/2018", "total": "4.90"} 

X51005230616 
  gold : {'company': 'GERBANG ALAF RESTAURANTS SDN BHD', 'date': '18/01/2018', 'total': '38.90'} 
  base : {"company": "190", "date": "18/01/2018", "total": "50.00"} 

In [32]:
# Where does the fine-tuned model fail? Print the receipts it gets wrong, with the failing field.
n_bad = 0
for k, f in ft.items():
    s = score_example(f["pred"], f["gold"])
    bad = [x for x in FIELDS if not s[x]["em"]]
    if bad:
        n_bad += 1
        if n_bad <= 10:
            print(k, "| unseen merchant:", not f["seen_merchant"], "| wrong:", bad)
            for x in bad:
                print(f"   {x}: gold={f['gold'][x]!r} pred={(f['pred'] or {}).get(x)!r} grounded={(f['grounded'] or {}).get(x)}")
print("receipts with >=1 wrong field:", n_bad, "of", len(ft))

X51005230605 | unseen merchant: True | wrong: ['company']
   company: gold='PETRON BKT LANJAN SB' pred='ALSERKAM ENTERPRISE' grounded=True
X51005268275 | unseen merchant: False | wrong: ['total']
   total: gold='278.80' pred='263.02' grounded=True
X51005337867 | unseen merchant: True | wrong: ['company', 'total']
   company: gold='OLD TOWN KOPITAM SND BHD' pred='OLDTOWN KOPITIAM SDN BHD' grounded=True
   total: gold='30.25' pred='30.24' grounded=True
X51005361906 | unseen merchant: False | wrong: ['company']
   company: gold='TED HENG STATIONERY & BOOKS' pred='TEO HENG STATIONERY & BOOKS' grounded=True
X51005442322 | unseen merchant: True | wrong: ['date']
   date: gold='03 MAR 19' pred='03 MAR 18' grounded=True
X51005442366 | unseen merchant: False | wrong: ['company']
   company: gold='TRIPLE SIX POINT ENTERPRISE 666' pred='TRIPLE SIX POINT ENTERPRISE' grounded=True
X51005568866 | unseen merchant: False | wrong: ['date']
   date: gold='20/08/17' pred='25 AUG - 3 SEPT 2017' grounded=T

## 6. Export for edge deployment (optional)
Merge the adapter into the base model, then convert the merged HF checkpoint with llama.cpp (GGUF, Q4_K_M) or MLX on the Mac. Report size and latency of the quantised model in the write-up.

In [33]:
!python -m docextract.export --model Qwen/Qwen3-0.6B --adapter runs/qwen3-0.6b_lora_r16/adapter --out merged/qwen3-0.6b_lora_r16
# GGUF (llama.cpp):
# !git clone -q https://github.com/ggml-org/llama.cpp && pip -q install -r llama.cpp/requirements.txt
# !python llama.cpp/convert_hf_to_gguf.py merged/qwen3-0.6b_lora_r16 --outtype q8_0 --outfile merged/qwen3-0.6b_lora_r16.q8_0.gguf
# On the Mac (MLX): mlx_lm.convert --hf-path merged/qwen3-0.6b_lora_r16 -q

Loading weights: 100% 311/311 [00:00<00:00, 8405.15it/s]
Writing model shards: 100% 1/1 [00:04<00:00,  4.11s/it]
saved merged model to merged/qwen3-0.6b_lora_r16


## 7. Save artefacts
Freeze the environment (this is the lock file for the README) and download the small outputs to commit.

In [34]:
!pip freeze | grep -iE "^(torch|transformers|peft|datasets|accelerate|bitsandbytes)==" > results/env_freeze.txt
!python --version >> results/env_freeze.txt
!nvidia-smi -L >> results/env_freeze.txt
!cat results/env_freeze.txt
!zip -qr artifacts.zip preds results runs -x "runs/*/checkpoints/*" "runs/*/adapter/*" "runs/smoke/*"
from google.colab import files
files.download("artifacts.zip")

accelerate==1.15.0
bitsandbytes==0.50.2
datasets==4.8.5
peft==0.21.1
torch==2.11.0+cu130
transformers==5.18.0
Python 3.13.15
GPU 0: Tesla T4 (UUID: GPU-3e25adc2-645a-b0b0-9f51-1994b76d9630)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>